# Ataxx Zero — Kaggle Training

**Setup obligatorio antes de Run All:**
1. Settings → Accelerator → **GPU T4 x2** (las dos T4, no P100).
2. Add-ons → Secrets → agregar `HF_TOKEN` (label `HF_TOKEN`, value el token).
3. Editar la celda **"Run config"** abajo para nombrar el run y elegir scope.

Toda la config vive en una sola celda. La celda de training escribe el config a JSON y llama a `train.py` via subprocess (multi-GPU `ddp_spawn` solo funciona si los workers re-ejecutan un script real, no el kernel del notebook).

In [ ]:
# === Run config (lo único que hay que tocar entre runs) ===

# Identidad del run
RUN_NAME = "policy_spatial_v8"
BOOTSTRAP_FROM = ""                       # "" = fresh run; ej "policy_spatial_v6" hereda pesos.
RESET_ITERATION = False                   # True solo con BOOTSTRAP_FROM != "". Requiere WARMUP > 0 (guard PM04).

# Scope
ITERATIONS = 180
EPISODES_PER_ITER = 20
MCTS_SIMS = 160

# Modelo (arquitectura v6, lo mejor probado hasta ahora)
D_MODEL = 128
NUM_LAYERS = 6
NHEAD = 8
DIM_FEEDFORWARD = 512
DROPOUT = 0.1

# Optimizador
EPOCHS = 1
BATCH_SIZE = 224
LR = 3e-4
WEIGHT_DECAY = 1e-4
VALUE_LOSS_COEFF = 0.5

# Replay sampling
BUFFER_SIZE = 50_000
TRAIN_RECENT_FRACTION = 0.7
TRAIN_RECENT_WINDOW_FRACTION = 0.4

# Eval (post-PM04: 64 partidas → SE ~0.06, delta=0.06 estadísticamente significativo)
EVAL_EVERY = 6
EVAL_GAMES = 64
EVAL_SIMS = 160
EVAL_REGRESSION_DELTA = 0.06
EVAL_REGRESSION_PATIENCE = 2
RESTORE_BEST_ON_REGRESSION = True
EVAL_HEURISTIC_LEVELS = "easy,normal,hard,apex,gambit,sentinel"   # 6 niveles → perfil completo en metadata HF.
SAVE_EVERY = EVAL_EVERY                                           # alinear con EVAL_EVERY: cada eval queda persistido en metadata.

# Warmup (recomendado: >0 siempre; obligatorio si BOOTSTRAP_FROM + RESET_ITERATION)
WARMUP_GAMES = 320
WARMUP_EPOCHS = 4

# Mezcla de oponentes en self-play (probabilidades — se normalizan)
OPPONENT_SELF_PROB = 0.45
OPPONENT_HEURISTIC_PROB = 0.50
OPPONENT_RANDOM_PROB = 0.05

# Hardware (Kaggle T4 x2)
TRAINER_DEVICES = 2
TRAINER_STRATEGY = "ddp_spawn"
TRAINER_PRECISION = "16-mixed"
SELFPLAY_WORKERS = 2

# Persistencia HF
HF_REPO_ID = "dieg0code/ataxx-zero"
KEEP_LAST_N_LOCAL = 3

In [ ]:
# === Clonar / actualizar el repo ===

REPO_URL = "https://github.com/Dieg0Code/ataxx-zero-ai.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/ataxx-zero-ai"

import os, subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.check_call(["git", "clone", REPO_URL, WORKDIR])
else:
    subprocess.check_call(["git", "-C", WORKDIR, "fetch", "--prune"])
subprocess.check_call(["git", "-C", WORKDIR, "checkout", BRANCH])
subprocess.check_call(["git", "-C", WORKDIR, "pull", "--ff-only"])

os.chdir(WORKDIR)
subprocess.check_call(["git", "-C", WORKDIR, "log", "-1", "--oneline"])

In [ ]:
# === Instalar dependencias (uv + grupo train) ===

!python -m pip install -q uv
!uv sync --frozen --group train
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())"

In [ ]:
# === HF Token desde Kaggle Secrets ===

HF_ENABLED = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    if not token:
        raise RuntimeError("HF_TOKEN secret está vacío.")
    os.environ["HF_TOKEN"] = token
    HF_ENABLED = True
    print(f"HF_TOKEN cargado ({len(token)} chars). HF persistence ENABLED.")
except Exception as exc:
    print(f"Sin HF_TOKEN en Kaggle Secrets: {exc}")
    print("HF persistence DISABLED — los checkpoints solo van a quedar locales y se pierden al cerrar la sesión.")

In [ ]:
# === Detectar GPUs y armar el config JSON ===

import json, torch

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
    print(f"GPUs detectadas: {n_gpus} ({', '.join(names)})")
    if n_gpus < TRAINER_DEVICES:
        print(f"  ⚠️  Pediste TRAINER_DEVICES={TRAINER_DEVICES} pero hay {n_gpus}. Ajustando.")
        effective_devices = n_gpus
        effective_strategy = "auto" if n_gpus == 1 else TRAINER_STRATEGY
    else:
        effective_devices = TRAINER_DEVICES
        effective_strategy = TRAINER_STRATEGY if TRAINER_DEVICES > 1 else "auto"
else:
    print("⚠️  Sin CUDA — corriendo en CPU. Va a ser MUY lento (cambiá Accelerator en Settings).")
    effective_devices = 1
    effective_strategy = "auto"

run_config = {
    "iterations": ITERATIONS,
    "episodes_per_iter": EPISODES_PER_ITER,
    "mcts_sims": MCTS_SIMS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "value_loss_coeff": VALUE_LOSS_COEFF,
    "d_model": D_MODEL,
    "num_layers": NUM_LAYERS,
    "nhead": NHEAD,
    "dim_feedforward": DIM_FEEDFORWARD,
    "dropout": DROPOUT,
    "buffer_size": BUFFER_SIZE,
    "train_recent_fraction": TRAIN_RECENT_FRACTION,
    "train_recent_window_fraction": TRAIN_RECENT_WINDOW_FRACTION,
    "eval_every": EVAL_EVERY,
    "eval_games": EVAL_GAMES,
    "eval_sims": EVAL_SIMS,
    "eval_regression_delta": EVAL_REGRESSION_DELTA,
    "eval_heuristic_levels": EVAL_HEURISTIC_LEVELS,
    "save_every": SAVE_EVERY,
    "eval_regression_patience": EVAL_REGRESSION_PATIENCE,
    "restore_best_on_regression": RESTORE_BEST_ON_REGRESSION,
    "warmup_games": WARMUP_GAMES,
    "warmup_epochs": WARMUP_EPOCHS,
    "opponent_self_prob": OPPONENT_SELF_PROB,
    "opponent_heuristic_prob": OPPONENT_HEURISTIC_PROB,
    "opponent_random_prob": OPPONENT_RANDOM_PROB,
    "trainer_devices": effective_devices,
    "trainer_strategy": effective_strategy,
    "trainer_precision": TRAINER_PRECISION,
    "selfplay_workers": SELFPLAY_WORKERS,
    "log_dir": "/kaggle/working/logs",
    "checkpoint_dir": "/kaggle/working/checkpoints",
    "hf_enabled": HF_ENABLED,
    "hf_repo_id": HF_REPO_ID,
    "hf_run_id": RUN_NAME,
    "hf_bootstrap_run_id": BOOTSTRAP_FROM,
    "hf_reset_iteration": RESET_ITERATION,
    "hf_local_dir": "/kaggle/working/hf_checkpoints",
    "keep_last_n_local_checkpoints": KEEP_LAST_N_LOCAL,
    "keep_last_n_hf_checkpoints": KEEP_LAST_N_LOCAL,
}

config_path = "/kaggle/working/run_config.json"
with open(config_path, "w", encoding="utf-8") as fh:
    json.dump(run_config, fh, indent=2)

se_estimate = (0.5 / max(1, EVAL_GAMES) ** 0.5)
print(f"\n— Run '{RUN_NAME}' —")
print(f"  bootstrap     : {BOOTSTRAP_FROM or '(fresh run)'}{' + reset' if (BOOTSTRAP_FROM and RESET_ITERATION) else ''}")
print(f"  scope         : {ITERATIONS} iters × {EPISODES_PER_ITER} eps × {MCTS_SIMS} sims")
print(f"  model         : d_model={D_MODEL}, layers={NUM_LAYERS}, heads={NHEAD}")
print(f"  hardware      : devices={effective_devices}, strategy={effective_strategy}, precision={TRAINER_PRECISION}")
print(f"  eval          : every={EVAL_EVERY} iters × {EVAL_GAMES} games (SE≈{se_estimate:.3f}), delta={EVAL_REGRESSION_DELTA}")
print(f"  warmup        : {WARMUP_GAMES} games × {WARMUP_EPOCHS} epochs")
print(f"  hf persistence: {'ENABLED → ' + HF_REPO_ID if HF_ENABLED else 'DISABLED'}")
print(f"  config json   : {config_path}")

In [ ]:
# === Entrenar ===
# train.py corre como subprocess (no `import train`) para que ddp_spawn pueda
# re-ejecutar el script en cada worker. Llamar train.main() directo desde el
# notebook hace que los workers spawned pidan re-ejecutar el kernel launcher
# de Jupyter, que rechaza los args de Lightning con SystemExit: 2.

extra = ["--hf"] if HF_ENABLED else []
subprocess.check_call([
    "uv", "run", "python", "train.py",
    "--config-json", config_path,
    *extra,
])

In [ ]:
# === Post-run: bajar historial a CSV ===

if HF_ENABLED:
    !uv run python scripts/fetch_run_history.py {RUN_NAME}
    print(f"\nCSV en runs_history/{RUN_NAME}/{RUN_NAME}_history.csv")
else:
    print("HF_ENABLED=False — no hay metadata en HF para descargar.")